In [ ]:
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
from sklearn.model_selection import train_test_split

**?????? ??????? ???????? ????????? ?????:**

*   label encoding ??? ??????? ??????????
*   ????????? ?????????????? ?????
*   ??????? ???? ? ??????? ? ???????????? ??????? ? ??????? tf-df
*   ???????????? ? ??????? Random Forest:
*      ?????? ????? ?? ????? ????????
*      ?????? ??????????? ??????????????? ?? ????? ????????


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

???????? ???  ??????? (200 ???????? ) ??? ???????????????? ?  ??? ????? ( 50 ????????). ??? ?? ? ??????????? ??????? ???? ????????? ?????????????? ????????, ????? ???:
*   Year (???)
*   Month
*   Day
*   Day of week
*   season

?????? ?????????? (topic)???? ???????????? label encording





In [ ]:
# df = pd.read_csv('/content/drive/Othercomputers/My Laptop/Project_1_HSE/clear_data.csv')
# df.drop('Unnamed: 0', axis=1, inplace=True)

X_train = pd.read_csv('/content/drive/Othercomputers/My Laptop/Project_1_HSE/??????/train_200k_texts.csv')
##Y_train = pd.read_csv('/content/drive/Othercomputers/My Laptop/Project_1_HSE/??????/ytrain_200k_texts.csv')
X_test = pd.read_csv('/content/drive/Othercomputers/My Laptop/Project_1_HSE/??????/test_50k_texts.csv')
#Y_test = pd.read_csv('/content/drive/Othercomputers/My Laptop/Project_1_HSE/??????/ytest_50k_texts.csv')

In [ ]:
X_train

In [ ]:
X_train['topic'].value_counts()

In [ ]:
X_test['topic'].value_counts()

In [ ]:
X_train.shape

In [ ]:
X_test.shape

??? ??? ??? ?????? ('??????', '??????????') ????? ??????????? ? ????????? ?? ?????? ?? ?????????? - ?????? ??.

In [ ]:
for topic in ['??????', '??????????']:
  X_train = X_train.drop(X_train[X_train['topic'] == topic].index)
  X_test = X_test.drop(X_test[X_test['topic'] == topic].index)

X_train.reset_index(inplace=True)
X_test.reset_index(inplace=True)

In [ ]:
X_train

In [ ]:
Y_train = X_train['topic_le']
Y_test = X_test['topic_le']
Y_train.value_counts()

In [ ]:
date_cols = ['year', 'day_of_week', 'seanos']

???????? ????? ????????? ??? ???????? ????????? ? ????? ?????????? ?? OHE ? ????????? ? ????? ???????

In [ ]:
X_train_num = X_train[date_cols]
X_test_num = X_test[date_cols]
X_train_num

In [ ]:
enc = OneHotEncoder(drop='first', handle_unknown='error', sparse=True)

enc.fit(X_train_num)

train_num = enc.transform(X_train_num)
#feature_names = enc.get_feature_names(columns_cat)
test_num = enc.transform(X_test_num)

In [ ]:
train_num

**Text data transformation** 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1,1), min_df = 0.0001, max_df = 0.8)
X_train_bow = vec.fit_transform(X_train['text_str'])

In [ ]:
X_test_bow = vec.transform(X_test['text_str'])

In [ ]:
X_train_bow

???????? ??? ???????  - ? ????????? ?????????? ? ? ??????????????? ??????????.

In [ ]:
full_X_train = scipy.sparse.hstack([X_train_bow, train_num]).tocsr()
full_X_test = scipy.sparse.hstack([X_test_bow, test_num]).tocsr()

In [ ]:
full_X_train

????? ???????? ?? ??????? ?????? ????? ?????????? ??? ?????? ?????????? ????. ?? ? ???? ???????? ???????? ??? ??????? - ?? ?????????? . ?? ????? ????????. ????? ????????? ?????? ??????????????. ?? ??????? ???? ??? ????? ?? ?????? ??????.

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import GridSearchCV
# parameter_grid = {
#             'max_depth': [30,100,200],
#             'n_estimators': [500,1000],
#             'min_samples_split':[5,20]
#         }
# model = RandomForestClassifier(random_state=123, n_jobs=-1, verbose=2)
# grid_searcher = GridSearchCV(model, parameter_grid, cv=0, verbose=2)
# grid_searcher.fit(full_X_train, Y_train)
# model_best = grid_searcher.best_estimator_
 
# print('Best params = ', model_best.get_params())

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

best_model = None
best_score = 0
best_params = ()

#for max_depth in [30, 100, 200]:
for max_depth in [100,200]:
  for n_estimators in [500]:
    for min_samples_split in [5]:
      print(f'====== start training for max_depth: {max_depth}, n_estimators: {n_estimators}, min_samples_split: {min_samples_split}')
      model = RandomForestClassifier(
        max_depth=max_depth,
        n_estimators=n_estimators,
        min_samples_split=min_samples_split,
        random_state=123, 
        n_jobs=-1, 
        verbose=1
      )
      model.fit(full_X_train, Y_train)
      pred = model.predict(full_X_test)
      score = f1_score(pred, Y_test, sample_weight=None, average="micro")
      print(f'====== finished training for max_depth: {max_depth}, n_estimators: {n_estimators}, min_samples_split: {min_samples_split}, score: {score}')
      if score > best_score:
        best_score = score
        best_model = model
        best_params = (max_depth, n_estimators, min_samples_split)

print(f'Done. Best score is {best_score}, params {best_params}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ??????? ?????? ???? ?? 300 ???????? ???????? 100
model = RandomForestClassifier(
  n_estimators=300,
  max_depth = 100,
  #min_samples_split = 5,
  bootstrap=True,
  max_features='sqrt',
  verbose=2,
  n_jobs=-1
)

# ??????? ?? ????????????? ??????
model.fit(full_X_train, Y_train)

?? ????? ???? 0.74

In [ ]:
  from sklearn.metrics import f1_score
  pred = model.predict(full_X_test)
  score = f1_score(pred, Y_test, sample_weight=None, average="micro")
  score

In [ ]:
import pickle
with open('./model2.pkl', 'wb') as fp:
  pickle.dump(model, fp)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ??????? ?????? ???? ?? 400 ???????? ? ???????? 60
model = RandomForestClassifier(
  n_estimators=400,
  max_depth = 60,
  #min_samples_split = 5,
  bootstrap=True,
  max_features='sqrt',
  verbose=2,
  n_jobs=-1
)

# ??????? ?? ????????????? ??????
model.fit(full_X_train, Y_train)

?? ?????? ???? 0.96. ?????? ?????? ????? ?????????????

In [ ]:
from sklearn.metrics import f1_score
pred_train = model.predict(full_X_train)
score_train = f1_score(pred_train, Y_train, sample_weight=None, average="micro")
score_train

?? ????? ???? - 0.73 ? ???? ?????? ?????? ??????? ??????????? ?? ?????

In [ ]:

pred = model.predict(full_X_test)
score = f1_score(pred, Y_test, sample_weight=None, average="micro")
score

In [ ]:
dict_topic = dict(zip(X_train.topic, X_train.topic_le))
keys = list(dict_topic.keys())
keys.sort()
dict_topic_new = {}
for key in keys:
  dict_topic_new[key] = dict_topic[key]
dict_topic_new

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn import metrics

print (metrics.classification_report (Y_train, pred_train,target_names = dict_topic))

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn import metrics

print (metrics.classification_report(Y_test, pred,target_names = dict_topic))

In [ ]:
import pickle
with open('./model3.pkl', 'wb') as fp:
  pickle.dump(model, fp)

????????? ?? ?? ?????????????? ?? ???????? ????? ? ??????? tf-df. ?? ???? ?????? ?????? - ??? ???????? ??????? ??????????? ?????? ??????.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ??????? ?????? ???? ?? 400 ????????
model = RandomForestClassifier(
  n_estimators=400,
  max_depth = 60,
  #min_samples_split = 5,
  bootstrap=True,
  max_features='sqrt',
  random_state=123,
  verbose=2,
  n_jobs=-1
)

# ??????? ?? ????????????? ??????
model.fit(full_X_train, Y_train)

In [ ]:
dict_topic = dict(zip(X_train.topic, X_train.topic_le))
keys = list(dict_topic.keys())
keys.sort()
dict_topic_new = {}
for key in keys:
  dict_topic_new[key] = dict_topic[key]
dict_topic_new

In [ ]:
from sklearn.metrics import f1_score
pred_train = model.predict(full_X_train)
score_train = f1_score(pred_train, Y_train, sample_weight=None, average="micro")
score_train

In [ ]:
pred = model.predict(full_X_test)
score = f1_score(pred, Y_test, sample_weight=None, average="micro")
score

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn import metrics

print (metrics.classification_report (Y_train, pred_train,target_names = dict_topic_new))

In [ ]:
print (metrics.classification_report(Y_test, pred,target_names = dict_topic_new))

???????? ???? ?? ???????? ?????? - 0.74

?????


*   Random forest - ??????,????????? ??????? ??????????????? ????????.
*   ????? ??????? ???? - ????? ???????? ??? ????????????????, ?? ??? ?? ???????? ??????? ?????????? ??????. ? ??????? ?? GPU random forest ?? ????????????.
     ?????? ?????? ????????? ?? 30 ???  - 1.5 ?????. 
